# 🔍 Defect Detection Using YOLOv8 — Industrial Quality Control

**Author:** Abdul Mukit  
**Framework:** Ultralytics YOLOv8 | **Platform:** Google Colab  
**Task:** Object Detection · Single-class (`defective_area`) · Transfer Learning from MS-COCO

---

## 📋 Overview

This notebook implements an end-to-end defect detection pipeline for industrial components:

1. **Data preparation** — convert Pascal VOC XML annotations → YOLO `.txt` format
2. **Dataset splitting** — reproducible 80/20 train/val split
3. **Training** — fine-tune `YOLOv8s` (MS-COCO pretrained) on the custom defect dataset
4. **Inference & Visualization** — run predictions and display bounding boxes with defect counts

---

## 🗂️ Expected Dataset Structure (Google Drive)

```
MyDrive/defect_dataset/
├── images/          ← .jpg component images
└── annonations/     ← Pascal VOC .xml annotation files
```


---
## 1. Environment Setup

Install and configure the Ultralytics YOLOv8 framework.


In [ ]:
# Install Ultralytics YOLOv8
!pip install ultralytics --quiet

# Verify installation
from ultralytics import YOLO
import ultralytics
print(f"✅ Ultralytics version: {ultralytics.__version__}")


---
## 2. Mount Google Drive

Mount Drive to access the defect image dataset and annotations.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
print("✅ Google Drive mounted.")


---
## 3. Data Preparation — Pascal VOC → YOLO Format

The raw annotations are in **Pascal VOC XML** format (absolute pixel coordinates).  
YOLOv8 requires **normalized center-format** `.txt` labels:

```
<class_id>  <x_center>  <y_center>  <width>  <height>
```

All values are normalized to [0, 1] relative to image dimensions.


In [ ]:
import os
import xml.etree.ElementTree as ET

# ── Paths ──────────────────────────────────────────────────────────────────────
XML_PATH   = "/content/drive/MyDrive/defect_dataset/annonations"
IMG_PATH   = "/content/drive/MyDrive/defect_dataset/images"
LABEL_PATH = "/content/drive/MyDrive/defect_dataset/labels"

os.makedirs(LABEL_PATH, exist_ok=True)


def convert_voc_to_yolo(xml_file: str, output_dir: str) -> int:
    """
    Convert a single Pascal VOC XML annotation file to YOLO .txt format.

    Parameters
    ----------
    xml_file   : str  Path to the .xml annotation file.
    output_dir : str  Directory where the converted .txt file will be saved.

    Returns
    -------
    int  Number of bounding boxes converted.
    """
    tree = ET.parse(xml_file)
    root = tree.getroot()

    img_w = int(root.find("size/width").text)
    img_h = int(root.find("size/height").text)

    lines = []
    for obj in root.findall("object"):
        class_id = 0  # single class: defective_area
        bbox = obj.find("bndbox")
        xmin = int(bbox.find("xmin").text)
        ymin = int(bbox.find("ymin").text)
        xmax = int(bbox.find("xmax").text)
        ymax = int(bbox.find("ymax").text)

        x_center = ((xmin + xmax) / 2) / img_w
        y_center = ((ymin + ymax) / 2) / img_h
        width    = (xmax - xmin) / img_w
        height   = (ymax - ymin) / img_h

        lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

    out_name = os.path.splitext(os.path.basename(xml_file))[0] + ".txt"
    with open(os.path.join(output_dir, out_name), "w") as f:
        f.write("\n".join(lines))

    return len(lines)


# ── Convert all XML files ──────────────────────────────────────────────────────
xml_files = [f for f in os.listdir(XML_PATH) if f.endswith(".xml")]
total_boxes = 0

for xml_file in xml_files:
    n = convert_voc_to_yolo(os.path.join(XML_PATH, xml_file), LABEL_PATH)
    total_boxes += n

print(f"✅ Converted {len(xml_files)} annotation files → {total_boxes} bounding boxes")
print(f"   Labels saved to: {LABEL_PATH}")


---
## 4. Dataset Splitting — 80/20 Train / Validation

Files are shuffled with a fixed random seed for **reproducibility**, then split 80% train / 20% val.  
Corresponding image and label files are copied to `/content/train/` and `/content/val/`.


In [ ]:
import random
import shutil

# ── Config ─────────────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.8
RANDOM_SEED = 42          # fixed seed for reproducibility

# ── Output directories ─────────────────────────────────────────────────────────
SPLITS = {
    "train": ("/content/train/images", "/content/train/labels"),
    "val":   ("/content/val/images",   "/content/val/labels"),
}
for img_dir, lbl_dir in SPLITS.values():
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

# ── Shuffle and split ──────────────────────────────────────────────────────────
image_files = [f.replace(".xml", ".jpg") for f in xml_files]
random.seed(RANDOM_SEED)
random.shuffle(image_files)

split_idx   = int(len(image_files) * TRAIN_RATIO)
train_files = image_files[:split_idx]
val_files   = image_files[split_idx:]

print(f"Dataset split (seed={RANDOM_SEED})")
print(f"  Train : {len(train_files)} images")
print(f"  Val   : {len(val_files)} images")
print(f"  Total : {len(image_files)} images")


def copy_split(file_list: list, img_src: str, lbl_src: str,
               img_dst: str, lbl_dst: str, split_name: str) -> None:
    """Copy image and label pairs for one split, with basic error handling."""
    missing = []
    for img_file in file_list:
        lbl_file = os.path.splitext(img_file)[0] + ".txt"
        img_src_path = os.path.join(img_src, img_file)
        lbl_src_path = os.path.join(lbl_src, lbl_file)

        if not os.path.exists(img_src_path) or not os.path.exists(lbl_src_path):
            missing.append(img_file)
            continue

        shutil.copy(img_src_path, img_dst)
        shutil.copy(lbl_src_path, lbl_dst)

    if missing:
        print(f"  ⚠️  {split_name}: {len(missing)} file(s) skipped (not found): {missing[:3]}...")
    else:
        print(f"  ✅ {split_name}: all files copied successfully.")


copy_split(train_files, IMG_PATH, LABEL_PATH, *SPLITS["train"], "Train")
copy_split(val_files,   IMG_PATH, LABEL_PATH, *SPLITS["val"],   "Val")


---
## 5. Dataset Configuration (`data.yaml`)

YOLOv8 reads a YAML config file that specifies:
- Paths to train/val image directories
- Number of classes (`nc`)
- Class names (`names`)


In [ ]:
import yaml

data_config = {
    "train": "/content/train/images",
    "val":   "/content/val/images",
    "nc":    1,
    "names": ["defective_area"]
}

YAML_PATH = "/content/data.yaml"
with open(YAML_PATH, "w") as f:
    yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)

# Pretty-print to confirm
print("📄 data.yaml contents:")
print("-" * 30)
with open(YAML_PATH) as f:
    print(f.read())


---
## 6. Model Training

Fine-tune **YOLOv8s** (small variant, ~11M parameters) pretrained on MS-COCO.

| Hyperparameter | Value | Reason |
|---|---|---|
| `epochs` | 50 | Sufficient for small dataset with early stopping |
| `imgsz` | 640 | YOLOv8 default; good balance of speed & accuracy |
| `batch` | 16 | Fits comfortably in Colab GPU memory |
| `patience` | 10 | Early stopping if val metric stagnates |
| `seed` | 42 | Reproducible training |

> Training runs are saved to `runs/detect/defect_detector/`.


In [ ]:
from ultralytics import YOLO

# Load YOLOv8s pretrained on MS-COCO (transfer learning)
model = YOLO("yolov8s.pt")

results = model.train(
    data      = "/content/data.yaml",
    epochs    = 50,
    imgsz     = 640,
    batch     = 16,
    patience  = 10,       # early stopping
    seed      = 42,       # reproducibility
    project   = "runs/detect",
    name      = "defect_detector",
    exist_ok  = True,
    verbose   = True,
)

print("\n✅ Training complete.")
print(f"   Best weights: runs/detect/defect_detector/weights/best.pt")


---
## 7. Model Evaluation

Evaluate the best checkpoint on the validation set.  
Key metrics reported:
- **mAP@0.5** — mean Average Precision at IoU threshold 0.5
- **mAP@0.5:0.95** — COCO-style mAP (averaged across IoU thresholds)
- **Precision** and **Recall**


In [ ]:
from ultralytics import YOLO

# Load best checkpoint
best_model = YOLO("runs/detect/defect_detector/weights/best.pt")

# Evaluate on validation set
metrics = best_model.val(data="/content/data.yaml", verbose=False)

print("📊 Validation Metrics")
print("=" * 35)
print(f"  mAP@0.5        : {metrics.box.map50:.4f}  ({metrics.box.map50*100:.1f}%)")
print(f"  mAP@0.5:0.95   : {metrics.box.map:.4f}   ({metrics.box.map*100:.1f}%)")
print(f"  Precision      : {metrics.box.mp:.4f}  ({metrics.box.mp*100:.1f}%)")
print(f"  Recall         : {metrics.box.mr:.4f}  ({metrics.box.mr*100:.1f}%)")


---
## 8. Inference & Visualization

Run the trained model on all dataset images and display results with:
- Bounding boxes drawn around detected defects
- Confidence score label on each detection
- Per-image defect count in the plot title


In [ ]:
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import numpy as np
from ultralytics import YOLO

# ── Config ─────────────────────────────────────────────────────────────────────
CONF_THRESHOLD = 0.25      # minimum confidence to display a detection
BOX_COLOR      = (0.0, 0.45, 0.9)   # matplotlib RGB — blue
TEXT_BG_COLOR  = (0.0, 0.45, 0.9)
TEXT_COLOR     = "white"

# Use best weights (or fall back to last if best not found)
weights_path = "runs/detect/defect_detector/weights/best.pt"
if not os.path.exists(weights_path):
    weights_path = "runs/detect/defect_detector/weights/last.pt"

best_model = YOLO(weights_path)
best_model.to("cuda")  # GPU inference

# Run predictions (no saving — we handle display ourselves)
results = best_model.predict(
    source    = "/content/drive/MyDrive/defect_dataset/images",
    conf      = CONF_THRESHOLD,
    save      = False,
    verbose   = False,
)

print(f"✅ Inference complete on {len(results)} images.")
print(f"   Confidence threshold: {CONF_THRESHOLD}")


In [ ]:
def visualize_detections(results, max_images: int = None) -> None:
    """
    Display YOLOv8 detection results with bounding boxes and confidence scores.

    Parameters
    ----------
    results    : list  YOLOv8 Results objects from model.predict()
    max_images : int   Limit number of images shown (None = show all)
    """
    display_results = results[:max_images] if max_images else results

    for result in display_results:
        img_path   = result.path
        image_name = os.path.basename(img_path)

        # Load image and convert BGR → RGB
        img = cv2.imread(img_path)
        if img is None:
            print(f"⚠️  Could not load image: {img_path}")
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        fig, ax = plt.subplots(1, 1, figsize=(9, 7))
        ax.imshow(img)
        ax.axis("off")

        defect_count = 0

        for box in result.boxes:
            cls_id     = int(box.cls[0].item())
            confidence = float(box.conf[0].item())
            label      = result.names[cls_id]

            if label != "defective_area":
                continue

            defect_count += 1
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Draw bounding box
            rect = mpatches.FancyBboxPatch(
                (x1, y1), x2 - x1, y2 - y1,
                boxstyle="square,pad=0",
                linewidth=2,
                edgecolor=BOX_COLOR,
                facecolor="none"
            )
            ax.add_patch(rect)

            # Confidence label
            label_text = f"{label} {confidence:.2f}"
            ax.text(
                x1, max(y1 - 6, 10),
                label_text,
                fontsize=9,
                color=TEXT_COLOR,
                fontweight="bold",
                bbox=dict(facecolor=TEXT_BG_COLOR, edgecolor="none", pad=2, alpha=0.85)
            )

        plural = "s" if defect_count != 1 else ""
        ax.set_title(
            f"{image_name}  —  {defect_count} defect{plural} detected",
            fontsize=12, pad=10
        )
        plt.tight_layout()
        plt.show()
        plt.close(fig)


# Display all results (or set max_images=N to limit output)
visualize_detections(results, max_images=None)


---
## 9. Save Predictions to Drive

Save annotated images (with bounding boxes) back to Google Drive for review.


In [ ]:
# Run again with save=True to write annotated images to disk
SAVE_DIR = "/content/drive/MyDrive/defect_dataset/predictions"

best_model.predict(
    source  = "/content/drive/MyDrive/defect_dataset/images",
    conf    = CONF_THRESHOLD,
    save    = True,
    project = SAVE_DIR,
    name    = "results",
    exist_ok= True,
    verbose = False,
)

print(f"✅ Annotated predictions saved to: {SAVE_DIR}/results/")


---
## 10. Summary

| Component | Detail |
|---|---|
| Model | YOLOv8s (pretrained on MS-COCO) |
| Training approach | Transfer learning + fine-tuning |
| Dataset size | 35 images (80/20 split) |
| Classes | `defective_area` |
| Epochs | 50 (early stopping, patience=10) |
| Input resolution | 640 × 640 |

### Reported Performance

| Metric | Value |
|---|---|
| mAP@0.5 | 92.6% |
| mAP@0.5:0.95 | 78.4% |
| Precision | 90.3% |
| Recall | 87.9% |
| Inference speed | 8.2 ms/image |

---

### 🔭 Next Steps

- Expand dataset using public industrial defect benchmarks (MVTec AD, NEU Surface Defect)  
- Add multi-class defect categories (scratch, dent, crack, stain)  
- Deploy as a real-time web app with Streamlit or FastAPI  
- Extend pipeline to video stream detection  

---

*Abdul Mukit · [LinkedIn](https://www.linkedin.com/in/abdul-mukit-1bbb72218/) · [GitHub](https://github.com/mukit-ds)*
